# Team-Only Data Preparation
This notebook processes raw team statistics CSVs to create a temporal dataset with historical lags, trends, and future targets.

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

print("Starting team-only data preparation...")

Starting team-only data preparation...


In [2]:
# Define team mapping and standardized codes
team_name_map = {
    'Atlanta Hawks': 'ATL',
    'Boston Celtics': 'BOS',
    'Brooklyn Nets': 'BKN',
    'Charlotte Hornets': 'CHA',
    'Chicago Bulls': 'CHI',
    'Cleveland Cavaliers': 'CLE',
    'Dallas Mavericks': 'DAL',
    'Denver Nuggets': 'DEN',
    'Detroit Pistons': 'DET',
    'Golden State Warriors': 'GSW',
    'Houston Rockets': 'HOU',
    'Indiana Pacers': 'IND',
    'Los Angeles Clippers': 'LAC',
    'Los Angeles Lakers': 'LAL',
    'Memphis Grizzlies': 'MEM',
    'Miami Heat': 'MIA',
    'Milwaukee Bucks': 'MIL',
    'Minnesota Timberwolves': 'MIN',
    'New Orleans Pelicans': 'NOP',
    'New York Knicks': 'NYK',
    'Oklahoma City Thunder': 'OKC',
    'Orlando Magic': 'ORL',
    'Philadelphia 76ers': 'PHI',
    'Phoenix Suns': 'PHX',
    'Portland Trail Blazers': 'POR',
    'Sacramento Kings': 'SAC',
    'San Antonio Spurs': 'SAS',
    'Toronto Raptors': 'TOR',
    'Utah Jazz': 'UTA',
    'Washington Wizards': 'WAS'
}

seasons = ['20_21', '21_22', '22_23', '23_24', '24_25']
season_labels = ['2020-21', '2021-22', '2022-23', '2023-24', '2024-25']

all_teams = []

for season, label in zip(seasons, season_labels):
    file_path = f'../data/raw/{season}_teams.csv'
    if not os.path.exists(file_path):
        continue
        
    df = pd.read_csv(file_path)
    df['season'] = label
    df['Team'] = df['Team'].str.replace('*', '', regex=False)
    df['team_code'] = df['Team'].map(team_name_map)
    
    # Standardize codes used in player data
    df['team_code'] = df['team_code'].replace({'BRK': 'BKN', 'CHO': 'CHA', 'PHO': 'PHX'})
    
    # Keep unique columns and rename
    df = df.loc[:, ~df.columns.duplicated()]
    
    cols = {
        'team_code': 'team',
        'season': 'season',
        'W': 'wins',
        'L': 'losses',
        'ORtg': 'ortg',
        'DRtg': 'drtg',
        'NRtg': 'nrtg',
        'Pace': 'pace',
        'TS%': 'ts_pct',
        'eFG%': 'efg_pct',
        'TOV%': 'tov_pct',
        'ORB%': 'orb_pct',
        'SOS': 'sos',
        'SRS': 'srs'
    }
    
    df_clean = df[list(cols.keys())].rename(columns=cols)
    df_clean = df_clean.dropna(subset=['team'])
    all_teams.append(df_clean)

df_teams = pd.concat(all_teams, ignore_index=True)
print(f"Loaded {len(df_teams)} total team records.")

Loaded 150 total team records.


In [3]:
def create_team_temporal_features(df):
    # Sort by team and season
    df = df.sort_values(['team', 'season'])
    
    temporal_data = []
    
    for team in df['team'].unique():
        team_df = df[df['team'] == team].copy()
        
        for i in range(len(team_df)):
            curr_row = team_df.iloc[i].to_dict()
            
            # Lag features (Wins, Ratings)
            for lag in range(1, 4):
                if i >= lag:
                    prev_row = team_df.iloc[i - lag]
                    curr_row[f'wins_lag{lag}'] = prev_row['wins']
                    curr_row[f'ortg_lag{lag}'] = prev_row['ortg']
                    curr_row[f'drtg_lag{lag}'] = prev_row['drtg']
                else:
                    curr_row[f'wins_lag{lag}'] = 0
                    curr_row[f'ortg_lag{lag}'] = 0
                    curr_row[f'drtg_lag{lag}'] = 0
            
            # Trends
            if i >= 1:
                curr_row['wins_trend_1yr'] = curr_row['wins'] - curr_row['wins_lag1']
            else:
                curr_row['wins_trend_1yr'] = 0
                
            if i >= 2:
                curr_row['wins_trend_2yr'] = curr_row['wins'] - curr_row['wins_lag2']
            else:
                curr_row['wins_trend_2yr'] = 0
                
            temporal_data.append(curr_row)
            
    return pd.DataFrame(temporal_data)

df_temporal = create_team_temporal_features(df_teams)
print(f"Created temporal features for teams. Shape: {df_temporal.shape}")

Created temporal features for teams. Shape: (150, 25)


In [4]:
def create_team_targets(df):
    df = df.sort_values(['team', 'season'])
    target_data = []
    
    for team in df['team'].unique():
        team_df = df[df['team'] == team].copy()
        
        for i in range(len(team_df)):
            curr_row = team_df.iloc[i].to_dict()
            
            # Target: Next Season Stats
            if i < len(team_df) - 1:
                next_row = team_df.iloc[i + 1]
                curr_row['target_next_wins'] = next_row['wins']
                curr_row['target_next_ortg'] = next_row['ortg']
                curr_row['target_next_drtg'] = next_row['drtg']
                curr_row['is_training'] = True
            else:
                # Latest season available (24-25), no target yet
                curr_row['target_next_wins'] = 0
                curr_row['target_next_ortg'] = 0
                curr_row['target_next_drtg'] = 0
                curr_row['is_training'] = False
                
            target_data.append(curr_row)
            
    return pd.DataFrame(target_data)

df_final = create_team_targets(df_temporal)
print(f"Created targets for teams. Final shape: {df_final.shape}")

output_path = '../data/processed/team_features_temporal.csv'
df_final.to_csv(output_path, index=False)
print(f"Saved team-only features to {output_path}")

Created targets for teams. Final shape: (150, 29)
Saved team-only features to ../data/processed/team_features_temporal.csv
